# Verificación manual
Ejecución de la lista de verificación manual del README sobre la interfaz de widgets. Cada paso usa los mismos botones de la interfaz.

In [1]:
import sys, json, re
sys.path.insert(0, "src")

from minichat.ui.chat_app import ChatApp
app = ChatApp("echo")
v = app.view
app.show()

In [2]:
def estado():
    return re.sub(r"<[^>]+>", "", v.status.value)

def enviar(rol, texto):
    v.role.value = rol
    v.text.value = texto
    v.btn_send.click()

def mostrar():
    print("Estado:", estado())
    for etiqueta, _ in v.sel_index.options:
        print("  ", etiqueta)

## 1. Enviar un mensaje con rol `user`

In [3]:
enviar("user", "hola, ¿cómo estás?")
mostrar()

Estado: Done.
   0: [02:19:31] user: hola, ¿cómo estás?
   1: [02:19:31] assistant: Interesting. Why do you think …


## 2. Agregar un mensaje `assistant` y otro `system`

In [4]:
enviar("assistant", "respuesta escrita a mano")
enviar("system", "eres un bot amable")
mostrar()

Estado: Messages updated.
   0: [02:19:31] user: hola, ¿cómo estás?
   1: [02:19:31] assistant: Interesting. Why do you think …
   2: [02:19:31] assistant: respuesta escrita a mano
   3: [02:19:31] system: eres un bot amable


## 3. Mover los sliders y enviar otro mensaje

In [5]:
v.temp.value, v.topp.value, v.topk.value, v.maxnt.value = 1.0, 0.5, 10, 64
enviar("user", "otro mensaje")
print(v._ctl.config)
mostrar()

GenerationConfig(temperature=1.0, top_p=0.5, top_k=10, max_new_tokens=64, max_ctx_tokens=900)
Estado: Done.
   0: [02:19:31] user: hola, ¿cómo estás?
   1: [02:19:31] assistant: Interesting. Why do you think …
   2: [02:19:31] assistant: respuesta escrita a mano
   3: [02:19:31] system: eres un bot amable
   4: [02:19:31] user: otro mensaje
   5: [02:19:31] assistant: You said: "otro mensaje". Tell…


## 4. Seleccionar un mensaje y borrarlo con **Delete selected**

In [6]:
v.sel_index.value = 0
v.btn_delete.click()
mostrar()

Estado: Messages updated.
   0: [02:19:31] assistant: Interesting. Why do you think …
   1: [02:19:31] assistant: respuesta escrita a mano
   2: [02:19:31] system: eres un bot amable
   3: [02:19:31] user: otro mensaje
   4: [02:19:31] assistant: You said: "otro mensaje". Tell…


## 5. **Export JSON**

In [7]:
v.btn_export.click()
print("Estado:", estado())
with open("chat_transcript.json", encoding="utf-8") as fh:
    for m in json.load(fh):
        print(f"{m['role']:>9}: {m['content']}  meta={m['meta']}")

Estado: Exported: chat_transcript.json
assistant: Interesting. Why do you think "hola, ¿cómo estás?"?  meta={'backend': 'echo', 'gen_cfg': {'temperature': 0.7, 'top_p': 0.95, 'top_k': 50, 'max_new_tokens': 120, 'max_ctx_tokens': 900}, 'latency_ms': 0, 'prompt_chars': 18, 'reply_chars': 51}
assistant: respuesta escrita a mano  meta={}
   system: eres un bot amable  meta={}
     user: otro mensaje  meta={}
assistant: You said: "otro mensaje". Tell me more.  meta={'backend': 'echo', 'gen_cfg': {'temperature': 1.0, 'top_p': 0.5, 'top_k': 10, 'max_new_tokens': 64, 'max_ctx_tokens': 900}, 'latency_ms': 0, 'prompt_chars': 12, 'reply_chars': 39}


## 6. **Delete ALL**

In [8]:
v.btn_clear_all.click()
mostrar()
print("Mensajes en la sesión:", len(v._session.all()))

Estado: Messages updated.
Mensajes en la sesión: 0
